<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">


# Python for Finance, 3rd Edition
## Chapter 14 · Statistics
&copy; Dr. Yves J. Hilpisch<br>
AI-supported by various LLMs<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh


## Notebook Goals
This notebook mirrors the code examples from the chapter in a Colab-ready
format so that you can run, tweak, and extend them interactively.


### How to Use This Notebook
- Run the cells top to bottom the first time to create all variables.
- Use additional cells for your own experiments or GenAI-assisted
  refactorings.
- Refer back to the book text for detailed explanations and context.


In [ ]:
from pathlib import Path
import subprocess
import sys

NOTEBOOK_SUBDIR = "notebooks"
COLAB_PACKAGES = {}
REPO_NAME = "py4fi3rd"
REPO_URL = "https://github.com/yhilpisch/py4fi3rd.git"


def _support_dir() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        support_dir = candidate / "notebooks"
        if (support_dir / "_book_notebook_support.py").exists():
            return support_dir
    if "google.colab" in sys.modules:
        root = Path("/content") / REPO_NAME
        if not root.exists():
            subprocess.run(
                ["git", "clone", "--depth", "1", REPO_URL, str(root)],
                check=True,
            )
        return root / "notebooks"
    raise RuntimeError("Could not locate notebook support helpers.")


SUPPORT_DIR = _support_dir()
if str(SUPPORT_DIR) not in sys.path:
    sys.path.insert(0, str(SUPPORT_DIR))

from _book_notebook_support import setup_notebook

CONTEXT = setup_notebook(
    notebook_subdir=NOTEBOOK_SUBDIR,
    colab_packages=COLAB_PACKAGES,
)

PROJECT_ROOT = CONTEXT["PROJECT_ROOT"]
NOTEBOOK_DIR = CONTEXT["NOTEBOOK_DIR"]
CODE_DIR = CONTEXT["CODE_DIR"]
CHAPTERS_DIR = CONTEXT["CHAPTERS_DIR"]
FIGURES_DIR = CONTEXT["FIGURES_DIR"]
DATA_DIR = CONTEXT["DATA_DIR"]

PROJECT_ROOT

Statistics underpins almost every quantitative finance task in this book:
modeling returns, diagnosing distributional assumptions, building diversified
portfolios, and interpreting model outputs and forecasts. This chapter
revisits core statistical concepts with practical Python examples that you can
extend in later, more specialised chapters. You start with simple diagnostics
for normality, using geometric Brownian motion as a benchmark model and then
confronting it with real market data. You then explore portfolio-level
statistics and efficient frontiers, before closing with a compact Bayesian
example that illustrates how to update beliefs in light of new evidence. The
machine-learning view of statistics is deferred to
<<ch_machine_and_deep_learning>>, where you build larger predictive models on
top of the tools introduced here.


## Normality Diagnostics


Many theoretical models in finance, from mean–variance portfolio theory to
classical option pricing, assume that returns are (log-)normally distributed.


### Benchmark: Geometric Brownian Motion


Geometric Brownian motion (GBM) is a canonical model for asset prices.


In [ ]:
# Import the plotting stack and apply the shared house style used in the
# figure scripts.
import math
from pathlib import Path
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
mpl.style.use("seaborn-v0_8")
mpl.rcParams.update({"font.family": "serif", "figure.dpi": 300})
# Create a random-number generator with a fixed seed for reproducible
# simulations.
rng = np.random.default_rng(seed=2027)
# Define the GBM parameters: initial level `s0`, short rate `r`, volatility
# `sigma`,
# horizon `T`, and number of paths `n_paths`.
s0 = 100.0
r = 0.02
sigma = 0.2
T = 1.0
n_paths = 250_000
# Draw standard-normal shocks for each simulated path.
z = rng.standard_normal(n_paths)
# Apply the closed-form GBM formula to obtain terminal index levels `s_T`.
s_T = s0 * np.exp((r - 0.5 * sigma**2) * T + sigma * math.sqrt(T) * z)
# Take natural logarithms to work with terminal log prices `log_s_T`.
log_s_T = np.log(s_T)
# Standardize the simulated log prices for a quick normality check.
log_s_T_std = (log_s_T - log_s_T.mean()) / log_s_T.std()

In [ ]:
from scipy import stats
# Plot a histogram and QQ plot for simulated GBM log levels.
mu_hat = float(log_s_T.mean())
sigma_hat = float(log_s_T.std(ddof=1))
fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.0))
ax = axes[0]
_, bins, _ = ax.hist(
    log_s_T,
    bins=80,
    density=True,
    color="tab:blue",
    alpha=0.7,
)
x_pdf = np.linspace(bins[0], bins[-1], 300)
ax.plot(
    x_pdf,
    stats.norm.pdf(x_pdf, loc=mu_hat, scale=sigma_hat),
    color="tab:red",
    linewidth=1.5,
    label="Fitted normal PDF",
)
ax.set_title("Histogram of Simulated Log Levels")
ax.set_xlabel("log $S_T$")
ax.set_ylabel("Density")
ax.grid(True, linestyle="--", alpha=0.3)
ax.legend(loc="best")
ax = axes[1]
stats.probplot(log_s_T, dist="norm", sparams=(mu_hat, sigma_hat), plot=ax)
ax.set_title("QQ Plot vs Fitted Normal")
ax.grid(True, linestyle="--", alpha=0.3)
fig.tight_layout()
plt.show()

In [ ]:
# Simulate short-horizon GBM log returns and compare them with a fitted
# normal law.
rng_returns = np.random.default_rng(seed=2027)
n_steps_returns = 50
dt_returns = T / n_steps_returns
shocks_returns = rng_returns.standard_normal((n_steps_returns, n_paths))
# Apply moment matching, as in the chapter figure script, before forming
# returns.
mean = shocks_returns.mean(axis=0, keepdims=True)
std = shocks_returns.std(axis=0, ddof=0, keepdims=True)
shocks_returns = (shocks_returns - mean) / std
drift_returns = (r - 0.5 * sigma**2) * dt_returns
log_returns_sim = (
    drift_returns + sigma * math.sqrt(dt_returns) * shocks_returns
)
log_returns_flat = log_returns_sim.ravel()
mu_step = float(log_returns_flat.mean())
sigma_step = float(log_returns_flat.std(ddof=1))
fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.0))
ax = axes[0]
_, bins, _ = ax.hist(
    log_returns_flat,
    bins=70,
    density=True,
    color="tab:blue",
    alpha=0.7,
    label="Simulated log returns",
)
x_pdf = np.linspace(bins[0], bins[-1], 300)
ax.plot(
    x_pdf,
    stats.norm.pdf(x_pdf, loc=mu_step, scale=sigma_step),
    color="tab:red",
    linewidth=1.5,
    label="Fitted normal PDF",
)
ax.set_title("Histogram of Simulated GBM Log Returns")
ax.set_xlabel("log return")
ax.set_ylabel("Density")
ax.grid(True, linestyle="--", alpha=0.3)
ax.legend(loc="upper left")
ax = axes[1]
stats.probplot(
    log_returns_flat,
    dist="norm",
    sparams=(mu_step, sigma_step),
    plot=ax,
)
ax.set_title("QQ Plot for Simulated GBM Log Returns")
ax.grid(True, linestyle="--", alpha=0.3)
fig.tight_layout()
plt.show()

### Real-World Return Distributions


Simulated GBM paths are convenient benchmarks, but real asset returns often
show heavier tails, skewness, and other deviations from normality.


In [ ]:
# Import `pandas` for tabular time-series handling.
import pandas as pd
# Load the chapter data relative to the notebook location.
data_path = Path("../data/eod_data.csv")
data = pd.read_csv(data_path, index_col="Date", parse_dates=True).dropna()
# Select four instruments: two technology stocks and two ETFs.
symbols = ["AAPL", "NVDA", "SPY", "GLD"]
# Subset the full dataset to these instruments and drop any remaining
# missing rows.
prices = data[symbols].dropna()
# Normalize all price series to start at 100 and plot them for a visual
# comparison.
(prices / prices.iloc[0] * 100).plot(figsize=(10, 6))

In [ ]:
# Compute daily log returns as the log of the price ratio between
# consecutive days.
log_returns = np.log(prices / prices.shift(1))
# Inspect the first few rows of the log-return series for a quick sanity check.
log_returns.head()

In [ ]:
# Plot per-asset histograms of daily log returns using 50 bins.
ax = log_returns.hist(bins=50, figsize=(10, 8))

In [ ]:
# Import the `scipy.stats` subpackage for statistical summaries and tests.
import scipy.stats as scs
# Define a helper that prints a small table of descriptive statistics for a
# 1D array.
def print_statistics(array: np.ndarray) -> None:
    """Print basic statistics for a 1D array."""
    # Use `scs.describe()` to compute size, min, max, mean, variance,
    # skewness, and
    # kurtosis in one call.
    sta = scs.describe(array, nan_policy="omit")
    # Print the statistics in a right-aligned table that is easy to scan.
    print(f"{'statistic':>14s} {'value':>15s}")
    print(30 * "-")
    print(f"{'size':>14s} {sta.nobs:15.0f}")
    print(f"{'min':>14s} {sta.minmax[0]:15.5f}")
    print(f"{'max':>14s} {sta.minmax[1]:15.5f}")
    print(f"{'mean':>14s} {sta.mean:15.5f}")
    print(f"{'std':>14s} {math.sqrt(sta.variance):15.5f}")
    print(f"{'skew':>14s} {sta.skewness:15.5f}")
    print(f"{'kurtosis':>14s} {sta.kurtosis:15.5f}")
# Define a helper that reports skewness, kurtosis, and the combined
# normality test
# p-value.
def normality_tests(arr: np.ndarray) -> None:
    """Print skewness, kurtosis, and combined normality test p-values."""
    # Convert the input to a NumPy array and drop any `NaN` values
    # before applying the
    # tests.
    arr = np.asarray(arr)
    arr = arr[~np.isnan(arr)]
    print(f"{'skew p-value':>14s} {scs.skewtest(arr)[1]:15.5f}")
    print(f"{'kurt p-value':>14s} {scs.kurtosistest(arr)[1]:15.5f}")
    print(f"{'norm p-value':>14s} {scs.normaltest(arr)[1]:15.5f}")
# Loop over all symbols, printing both descriptive statistics and normality
# tests for
# each return series.
for sym in symbols:
    print(f"\nResults for symbol {sym}")
    print(30 * "-")
    series = log_returns[sym].dropna()
    print_statistics(series.to_numpy())
    normality_tests(series.to_numpy())

In [ ]:
# Build the real-data diagnostics figure for SPY and AAPL in the same
# two-by-two layout as the chapter.
symbols_diag = ["SPY", "AAPL"]
prices_diag = data[symbols_diag].dropna()
log_returns_diag = np.log(prices_diag / prices_diag.shift(1)).dropna()
fig, axes = plt.subplots(2, 2, figsize=(9.5, 7.0))
for row, sym in enumerate(symbols_diag):
    series = log_returns_diag[sym].to_numpy()
    mu_hat = float(series.mean())
    sigma_hat = float(series.std(ddof=1))
    ax = axes[row, 0]
    _, bins, _ = ax.hist(
        series,
        bins=50,
        density=True,
        color="tab:blue",
        alpha=0.7,
        label=f"{sym} log returns",
    )
    x_pdf = np.linspace(bins[0], bins[-1], 300)
    ax.plot(
        x_pdf,
        stats.norm.pdf(x_pdf, loc=mu_hat, scale=sigma_hat),
        color="tab:red",
        linewidth=1.5,
        label="Fitted normal PDF",
    )
    ax.set_title(f"{sym}: Histogram and Fitted Normal")
    ax.set_xlabel("log return")
    ax.set_ylabel("Density")
    ax.grid(True, linestyle="--", alpha=0.3)
    ax.legend(loc="best")
    ax = axes[row, 1]
    stats.probplot(series, dist="norm", sparams=(mu_hat, sigma_hat), plot=ax)
    ax.set_title(f"{sym}: QQ Plot vs Normal")
    ax.grid(True, linestyle="--", alpha=0.3)
fig.tight_layout()
plt.show()

## Portfolio Statistics and Efficient Frontiers


Single-asset statistics are only a starting point.


### Data and Basic Portfolio Statistics


You continue with the same four instruments and their daily log returns.


In [ ]:
noa = len(symbols)
# Drop the initial `NaN` row of log returns to obtain a clean dataset.
rets = log_returns.dropna()
# Compute annualized expected returns and the matching covariance matrix.
mean_rets = rets.mean() * 252
cov_matrix = rets.cov() * 252

In [ ]:
mean_rets

In [ ]:
cov_matrix

In [ ]:
# Draw random non-negative portfolio weights for each asset.
weights = rng.random(noa)
# Normalize the weights so that they sum to 1 (100% invested).
weights /= np.sum(weights)
# Check that the weights indeed add up to 1.
weights, weights.sum()

In [ ]:
# Compute the annualized expected portfolio return as the dot product of
# weights and
# expected returns.
def port_ret(weights: np.ndarray) -> float:
    return float(np.sum(mean_rets * weights))
# Compute the portfolio volatility as the square root of the quadratic form
# $w^\top \Sigma w$.
def port_vol(weights: np.ndarray) -> float:
    return float(np.sqrt(weights.T @ cov_matrix @ weights))
# Evaluate both functions for a sample weight vector to obtain a risk-return
# point.
port_ret(weights), port_vol(weights)

### Random Portfolios and Sharpe Ratios


To visualize the risk–return trade-off, you can generate a large number of
random portfolios and plot expected volatility against expected return.


In [ ]:
# Choose how many random portfolios to sample.
n_portfolios = 2_500
# Allocate arrays to store expected returns and volatilities for each portfolio.
prets = np.empty(n_portfolios)
pvols = np.empty(n_portfolios)
# Continue with the same generator used for the preceding examples.
# Generate random long-only weights, compute their risk and return, and
# store the results.
for i in range(n_portfolios):
    w = rng.random(noa)
    w /= np.sum(w)
    prets[i] = port_ret(w)
    pvols[i] = port_vol(w)
# Plot the random-portfolio cloud colored by the Sharpe ratio.
fig, ax = plt.subplots(figsize=(7.5, 4.5))
scatter = ax.scatter(
    pvols,
    prets,
    c=prets / pvols,
    marker=".",
    alpha=0.8,
    cmap="coolwarm",
)
ax.set_xlabel("expected volatility")
ax.set_ylabel("expected return")
ax.grid(True, linestyle="--", alpha=0.3)
fig.colorbar(scatter, ax=ax).set_label("Sharpe ratio")
fig.tight_layout()
plt.show()

### Approximating the Efficient Frontier


You can obtain a smoother approximation to the efficient frontier by solving a
constrained optimization problem for a range of target returns.


In [ ]:
# Import `scipy.optimize` for constrained numerical optimization.
import scipy.optimize as sco
# Define a helper that returns the minimum-volatility weights for a target
# return.
def efficient_weights(
    target: float,
    initial: np.ndarray | None = None,
) -> np.ndarray:
    if initial is None:
        initial = np.repeat(1.0 / noa, noa)
    constraints = (
        {"type": "eq", "fun": lambda w: np.sum(w) - 1.0},
        {"type": "eq", "fun": lambda w: port_ret(w) - target},
    )
    bounds = tuple((0.0, 1.0) for _ in range(noa))
    result = sco.minimize(
        port_vol,
        initial,
        method="SLSQP",
        bounds=bounds,
        constraints=constraints,
    )
    if not result.success:
        msg = f"frontier optimization failed for target={target:.6f}"
        raise RuntimeError(msg)
    return np.asarray(result.x, dtype=float)

In [ ]:
# Compute the global minimum-variance portfolio first.
gmv = sco.minimize(
    port_vol,
    np.repeat(1.0 / noa, noa),
    method="SLSQP",
    bounds=tuple((0.0, 1.0) for _ in range(noa)),
    constraints=({"type": "eq", "fun": lambda w: np.sum(w) - 1.0},),
)
if not gmv.success:
    raise RuntimeError("global minimum-variance optimization failed")
gmv_weights = np.asarray(gmv.x, dtype=float)
gmv_return = port_ret(gmv_weights)
# Trace only the efficient branch, using warm starts for smoother results.
target_returns = np.linspace(gmv_return, prets.max(), 50)
frontier_weights = []
initial = gmv_weights
for target in target_returns:
    initial = efficient_weights(float(target), initial=initial)
    frontier_weights.append(initial)
frontier_weights = np.asarray(frontier_weights)
frontier_vols = np.array([port_vol(w) for w in frontier_weights], dtype=float)

In [ ]:
# Combine the random-portfolio cloud with the efficient frontier in the
# chapter figure style.
fig, ax = plt.subplots(figsize=(7.5, 4.5))
scatter = ax.scatter(
    pvols,
    prets,
    c=prets / pvols,
    marker=".",
    alpha=0.8,
    cmap="coolwarm",
    label="Random portfolios",
)
ax.plot(
    frontier_vols,
    target_returns,
    "b",
    linewidth=2.5,
    label="Efficient frontier",
)
ax.set_xlabel("expected volatility")
ax.set_ylabel("expected return")
ax.set_title("Random Portfolios and Efficient Frontier")
ax.grid(True, linestyle="--", alpha=0.3)
fig.colorbar(scatter, ax=ax).set_label("Sharpe ratio")
ax.legend(loc="best")
fig.tight_layout()
plt.show()

## Bayesian Updating in a Simple Example


Bayesian statistics provides a natural framework for reasoning about
uncertainty: you start from prior beliefs, observe data, and arrive at
posterior beliefs.


### Boxes, Hypotheses, and Bayes’ Rule


Imagine two boxes, $B_1$ and $B_2$, each filled with black and red balls.


In [ ]:
# Import `NumPy` for concise vector arithmetic.
import numpy as np
priors = np.array([0.5, 0.5])        # p(B1), p(B2)
like_black = np.array([30 / 90, 60 / 90])  # p(D | B1), p(D | B2)
evidence = np.sum(like_black * priors)     # p(D)
# Apply Bayes’ rule elementwise to obtain posterior probabilities for each
# hypothesis.
posteriors = like_black * priors / evidence
# Inspect the resulting posterior vector, which favors the box with more
# black balls.
posteriors

## Figure Generation (Optional)
Run the chapter's figure scripts under `code/figures/` to regenerate the PNG
files under `assets/figures/`.


In [ ]:
import runpy

scripts = [
    "../code/figures/ch14_gbm_log_returns_diagnostics.py",
    "../code/figures/ch14_normality_gbm.py",
    "../code/figures/ch14_random_portfolios_frontier.py",
    "../code/figures/ch14_realdata_returns_diagnostics.py",
]

for script in scripts:
    try:
        runpy.run_path(script, run_name="__main__")
        print(f"OK: {script}")
    except ModuleNotFoundError as e:
        print(f"Skipping {script}: missing dependency ({e.name}).")
    except Exception as e:
        print(f"Failed {script}: {type(e).__name__}: {e}")


<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">
